In [ ]:
import pandas as pd
import SimpleITK as sitk
import numpy as np
import seaborn as sns
import os
import matplotlib.pyplot as plt
import ast

In [ ]:
df_pre_data = pd.read_csv(r'preprocessing_log.csv')

In [ ]:
df_pre_data[df_pre_data['status']=='Failed'].to_csv('failed_preprocessing_temp.csv', index=False)

In [ ]:
df_pre_data.dtypes

In [ ]:
df_pre_data_suc = df_pre_data[df_pre_data['status']=='Success'].copy()

In [ ]:
df_pre_data_suc.shape

In [ ]:
df_pre_data_suc

In [ ]:
df_pre_data_suc['shape_z_y_x'] = df_pre_data_suc['shape_z_y_x'].apply(ast.literal_eval)

In [ ]:
def find_area(x):
    x,y,z= x
    return x*y*z

In [ ]:
df_pre_data_suc['area'] = df_pre_data_suc['shape_z_y_x'].apply(find_area)

In [ ]:
sns.boxplot(x=df_pre_data_suc['area'])

In [ ]:
list_of_files = os.listdir(r'processed_data')

In [7]:
train_df = pd.read_csv(r'rsna-intracranial-aneurysm-detection\train.csv')

In [10]:
train_df[train_df['Modality']=='CTA']['Aneurysm Present'].value_counts()

Aneurysm Present
1    997
0    860
Name: count, dtype: int64

In [ ]:
prep_df = pd.DataFrame((files[:-7] for files in list_of_files), columns=['SeriesInstanceUID'])
prep_df.iloc[3].values


In [ ]:
prep_df = prep_df.merge(train_df, how='inner', on='SeriesInstanceUID')
prep_df


In [ ]:
list_seg = os.listdir(r'rsna-intracranial-aneurysm-detection\segmentations')

In [ ]:
list_seg

In [ ]:
from view3d_data import *
from preprocess_ct import *

In [ ]:
seg_mask = sitk.ReadImage(str(r'rsna-intracranial-aneurysm-detection\segmentations\1.2.826.0.1.3680043.8.498.10076056930521523789588901704956188485\1.2.826.0.1.3680043.8.498.10076056930521523789588901704956188485.nii'))

In [ ]:
seg_mask_np = sitk.GetArrayFromImage(seg_mask)
seg_mask_np.shape

In [ ]:
view_3d_volume(seg_mask_np)

In [ ]:
import numpy as np

print("Unique values in mask:", np.unique(seg_mask_np))
print("Number of aneurysm voxels:", np.sum(seg_mask_np > 0))


In [ ]:
train_df[train_df['SeriesInstanceUID']=='1.2.826.0.1.3680043.8.498.10035643165968342618460849823699311381']

In [ ]:
local_df = pd.read_csv(r'rsna-intracranial-aneurysm-detection\train_localizers.csv')
local_df

In [1]:
import pandas as pd
df_new_logs = pd.read_csv(r'processed_data_v2\preprocessing_log.csv')
df_new_logs  = df_new_logs[df_new_logs['status']!='Skipped']
df_new_logs

,SeriesInstanceUID,status,shape_z_y_x,error
0,1.2.826.0.1.3680043.8.498.10035643165968342618...,Success,"(133, 349, 295)",NaN
1,1.2.826.0.1.3680043.8.498.10077108087009955586...,Success,"(167, 250, 394)",NaN
2,1.2.826.0.1.3680043.8.498.10192011262895867728...,Failed,NaN,need at least one array to stack
3,1.2.826.0.1.3680043.8.498.10291305271924252800...,Failed,NaN,need at least one array to stack
4,1.2.826.0.1.3680043.8.498.10311178483256099259...,Success,"(8, 135, 69)",NaN
...,...,...,...,...
389,1.2.826.0.1.3680043.8.498.99614492920407247894...,Success,"(178, 346, 272)",NaN
390,1.2.826.0.1.3680043.8.498.99639493469775227910...,Success,"(98, 325, 290)",NaN
391,1.2.826.0.1.3680043.8.498.99674090910456004499...,Success,"(143, 348, 266)",NaN
392,1.2.826.0.1.3680043.8.498.99887675554378211308...,Success,"(81, 231, 231)",NaN


In [ ]:
df_new_logs.to_csv(r'processed_data_v2\preprocessing_log.csv', index=False)

In [ ]:
df_new_logs = pd.read_csv(r'processed_data_v2\preprocessing_log.csv')
df_o_logs = pd.read_csv(r'processed_data_v2\preprocessing_log_1.csv')

In [ ]:
df_o_logs = df_o_logs[~df_o_logs['SeriesInstanceUID'].isin(df_new_logs['SeriesInstanceUID'])]

In [ ]:
df_o_logs[df_o_logs['SeriesInstanceUID'].isin(df_new_logs['SeriesInstanceUID'])]

In [ ]:
df_all_logs = pd.concat([df_new_logs, df_o_logs])
df_all_logs

In [ ]:
df_all_logs.to_csv('df_all_processing_logs.csv',index=False)

In [ ]:
df_new_localization = pd.read_csv(r'processed_data_v2\new_localization.csv')
df_old_localization = pd.read_csv(r'processed_data_v2\new_localization_1.csv')

In [ ]:
df_new_localization['SeriesInstanceUID'].value_counts()

In [ ]:
df_old_localization['SeriesInstanceUID'].value_counts()

In [ ]:
df_old_localization = df_old_localization[~df_old_localization['SeriesInstanceUID'].isin(df_new_localization['SeriesInstanceUID'])]
df_old_localization

In [ ]:
df_all_localization = pd.concat([df_new_localization, df_old_localization])
df_all_localization

In [ ]:
df_all_localization['SeriesInstanceUID'].nunique()

In [ ]:
df_all_localization.to_csv(r'df_all_localization.csv', index=False)

In [ ]:
df_train = pd.read_csv(r'rsna-intracranial-aneurysm-detection\train.csv')
df_train

In [ ]:
df_train[df_train['Modality']=='CTA']['Aneurysm Present'].value_counts()

In [ ]:
master_df = pd.merge(df_all_localization,df_train , how='left', on='SeriesInstanceUID')
master_df

In [ ]:
master_df['Aneurysm Present'].value_counts()

In [ ]:
df_train[df_train['Modality']=='CTA']['Aneurysm Present'].value_counts()

In [ ]:
master_df.to_csv(r'master_df_positive_ct.csv', index=False)

In [ ]:
# Step 1: split into columns
master_df[['coord_z', 'coord_y', 'coord_x']] = (
    master_df['final_coords_zyx']
    .str.strip("()")               # remove parentheses
    .str.split(",", expand=True)   # split into 3 columns
)

# Step 2: clean whitespace and stray characters
for col in ['coord_z', 'coord_y', 'coord_x']:
    master_df[col] = master_df[col].str.strip().str.replace(")", "").str.replace("(", "")

# Step 3: convert safely to numeric
master_df[['coord_z', 'coord_y', 'coord_x']] = master_df[['coord_z', 'coord_y', 'coord_x']].apply(
    pd.to_numeric, errors='coerce'   # invalid values -> NaN instead of error
)

# Optional: drop the original column
master_df.drop(columns=['final_coords_zyx'], inplace=True)

master_df.head()

In [ ]:
# master_df.drop(columns=['final_coords_zyx'], inplace=True)
master_df[['coord_z', 'coord_y', 'coord_x']] = master_df[['coord_z', 'coord_y', 'coord_x']].apply(pd.to_numeric)

In [ ]:
master_df.columns

In [ ]:
import seaborn as sns

In [ ]:
sns.pairplot(master_df[['coord_z', 'coord_y', 'coord_x' ]], diag_kind='kde')

In [ ]:
#read all seriesUID from traindf and add them to masterdf if the same seriesUID is not present and aneurysm present for that seriesUID is 0 in masterdf and set default values for all other columsn
master_df = pd.merge(df_train,df_all_localization, how='left', on='SeriesInstanceUID')

In [ ]:
neg_cases = df_train[(df_train['Aneurysm Present'] == 0) & (df_train['Modality'] == 'CTA')]
neg_cases

In [ ]:
master_df = pd.concat([master_df,neg_cases])
master_df

In [ ]:
master_df[master_df['Aneurysm Present'] == 0]

In [ ]:
master_df['SeriesInstanceUID'].nunique()

In [ ]:
# Save as check_data_consistency.py
import pandas as pd
import os

# --- CONFIGURE THESE ---
MASTER_CSV_PATH = 'master_df_positive_ct.csv'
NPY_DATA_DIR = 'processed_data_npy'
# ---------------------

print("--- Running Data Consistency Check ---")

# 1. Get all unique UIDs from the CSV file
try:
    df_master = pd.read_csv(MASTER_CSV_PATH)
    csv_uids = set(df_master['SeriesInstanceUID'].unique())
    print(f"Found {len(csv_uids)} unique SeriesInstanceUIDs in {MASTER_CSV_PATH}")
except FileNotFoundError:
    print(f"ERROR: Master CSV not found at {MASTER_CSV_PATH}")
    exit()

# 2. Get all UIDs from the .npy files on disk
try:
    # Get all filenames, remove the '.npy' extension to get the UID
    file_uids = set([f.replace('.npy', '') for f in os.listdir(NPY_DATA_DIR) if f.endswith('.npy')])
    print(f"Found {len(file_uids)} .npy files in {NPY_DATA_DIR}")
except FileNotFoundError:
    print(f"ERROR: NPY data directory not found at {NPY_DATA_DIR}")
    exit()

print("-" * 30)

# 3. Find the differences
files_not_in_csv = file_uids - csv_uids
csv_uids_not_in_files = csv_uids - file_uids

if not files_not_in_csv and not csv_uids_not_in_files:
    print("SUCCESS: Data is perfectly consistent!")
else:
    if files_not_in_csv:
        print(f"\nWARNING: Found {len(files_not_in_csv)} .npy files that are NOT listed in the master CSV:")
        for uid in files_not_in_csv:
            print(f" -> {uid}")
        print("(This is OK if these files are intentionally excluded, e.g., negative cases.)")

    if csv_uids_not_in_files:
        print(f"\nWARNING: Found {len(csv_uids_not_in_files)} UIDs in the CSV that are MISSING their .npy file:")
        for uid in csv_uids_not_in_files:
            print(f" -> {uid}")
        print("(This is expected for scans that failed during preprocessing.)")

print("\n--- Check Complete ---")

In [3]:
train_manif = pd.read_csv(r'aneurysm_dataset_manifests\test_manifest.csv')

In [5]:
train_manif['Aneurysm Present'].value_counts()

Aneurysm Present
0    15553
1      850
Name: count, dtype: int64

In [6]:
master_df = pd.read_csv(r'master_df_positive_ct.csv')
master_df['Aneurysm Present'].value_counts()

Aneurysm Present
1    1235
0     860
Name: count, dtype: int64

In [11]:
train_manif = pd.read_csv(r'aneurysm_dataset_manifests\train_manifest.csv')


In [12]:
train_manif['Aneurysm Present'].value_counts()


Aneurysm Present
0    108522
1      6896
Name: count, dtype: int64

In [14]:
# Save this as balance_dataset.py

import pandas as pd
import os

# --- 1. CONFIGURATION ---

# The directory where your train/test/val manifest CSVs are saved
MANIFEST_DIR = 'aneurysm_dataset_manifests' 

# The directory where the new, balanced manifests will be saved
OUTPUT_DIR = 'aneurysm_dataset_manifests_balanced'

# --- SAMPLING PARAMETERS ---
# We want a ratio of NEGATIVE_RATIO negatives for every 1 positive.
# 2.0 means a 2:1 ratio of negative to positive patches.
NEGATIVE_RATIO = 1.68

# --- 2. SCRIPT LOGIC ---

def balance_manifests():
    """
    Loads the generated manifest files, downsamples the negative cases in the
    training and validation sets, and saves new, balanced CSVs.
    The test set is left untouched to provide an unbiased evaluation.
    """
    print("--- Starting Manifest Balancing ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # We only balance the training and validation sets.
    # The test set must remain in its original, imbalanced distribution
    # to give a true measure of real-world performance.
    for split in ['train', 'validation', 'test']:
        input_path = os.path.join(MANIFEST_DIR, f"{split}_manifest.csv")
        output_path = os.path.join(OUTPUT_DIR, f"{split}_manifest.csv")

        try:
            df = pd.read_csv(input_path)
        except FileNotFoundError:
            print(f"Warning: Manifest file not found at {input_path}. Skipping.")
            continue

        print(f"\nProcessing {split}_manifest.csv...")
        
        if split in ['test','validation']:
            print(" -> Test set is not balanced. Copying as is.")
            df.to_csv(output_path, index=False)
            continue

        # Separate positive and negative patches
        df_pos = df[df['Aneurysm Present'] == 1]
        df_neg = df[df['Aneurysm Present'] == 0]

        n_pos = len(df_pos)
        n_neg_desired = int(n_pos * NEGATIVE_RATIO)

        print(f" -> Found {n_pos} positive patches.")
        print(f" -> Found {len(df_neg)} negative patches.")
        
        if len(df_neg) < n_neg_desired:
            print(f" -> Warning: Not enough negative samples to meet the desired ratio. Using all {len(df_neg)} negative samples.")
            df_neg_sampled = df_neg
        else:
            print(f" -> Sampling {n_neg_desired} negative patches...")
            # Randomly sample the negative patches
            df_neg_sampled = df_neg.sample(n=n_neg_desired, random_state=42)

        # Combine the positive and sampled negative patches
        df_balanced = pd.concat([df_pos, df_neg_sampled])
        
        # Shuffle the final dataframe to mix positive and negative samples
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

        print(f" -> Saving balanced manifest with {len(df_balanced)} total patches to {output_path}")
        df_balanced.to_csv(output_path, index=False)

    print("\n--- Balancing Complete ---")
    print(f"All balanced manifest files saved in: {OUTPUT_DIR}")

if __name__ == '__main__':
    balance_manifests()

--- Starting Manifest Balancing ---

Processing train_manifest.csv...
 -> Found 6896 positive patches.
 -> Found 108522 negative patches.
 -> Sampling 11585 negative patches...
 -> Saving balanced manifest with 18481 total patches to aneurysm_dataset_manifests_balanced\train_manifest.csv

Processing validation_manifest.csv...
 -> Test set is not balanced. Copying as is.

Processing test_manifest.csv...
 -> Test set is not balanced. Copying as is.

--- Balancing Complete ---
All balanced manifest files saved in: aneurysm_dataset_manifests_balanced


In [15]:
train_manif = pd.read_csv(r'aneurysm_dataset_manifests_balanced\train_manifest.csv')
train_manif['Aneurysm Present'].value_counts()

Aneurysm Present
0    11585
1     6896
Name: count, dtype: int64

In [17]:
import torch
from monai.bundle import ConfigParser

# --- Part 1: Analyze SegResNet ---
print("--- Analyzing SegResNet ---")
cfg_seg = ConfigParser()
cfg_seg.read_config("models/wholeBody_ct_segmentation/configs/inference.json")

# This line reads the config and creates the PyTorch model object
segresnet_model = cfg_seg.get_parsed_content("network_def")

# Print the entire model architecture
segresnet_model

--- Analyzing SegResNet ---


SegResNet(
  (act_mod): ReLU(inplace=True)
  (convInit): Convolution(
    (conv): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
  )
  (down_layers): ModuleList(
    (0): Sequential(
      (0): Identity()
      (1): ResBlock(
        (norm1): GroupNorm(8, 32, eps=1e-05, affine=True)
        (norm2): GroupNorm(8, 32, eps=1e-05, affine=True)
        (act): ReLU(inplace=True)
        (conv1): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        )
        (conv2): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        )
      )
    )
    (1): Sequential(
      (0): Convolution(
        (conv): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      )
      (1): ResBlock(
        (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
        (norm2): GroupNorm(8, 64, eps=1e-05, a

In [ ]:
# --- Part 2: Analyze Swin UNETR ---
print("\n\n--- Analyzing Swin UNETR ---")
cfg_swin = ConfigParser()
cfg_swin.read("models/swin_unetr_btcv_segmentation/configs/inference.json")

swin_model = cfg_swin.get_parsed_content("network")

swin_model